# Balanced Policy Gradient (BPG) on Hopper-v5

Reproduction of **Algorithm 1** from

> B. El Mabsout, A. Abdelgawad, R. Mancuso,
> *"Closing the intent-to-behavior gap via Fulfillment Priority Logic"*, **IROS 2025**.

**BPG** is **DDPG** restructured around *Fulfillment Priority Logic (FPL)*. The four changes over DDPG (highlighted in the paper) are:

1. **Fulfillment Q-values.** The critic predicts a *vector* `FQ = (1−γ)·Q ∈ [0,1]ⁿ`, one normalized value per objective (the discounted sum of rewards in `[0,1]` is bounded by `1/(1−γ)`, so the `(1−γ)` factor maps it back into `[0,1]`).
2. **FPL utility for the actor.** Instead of maximizing a scalar `Q`, the actor maximizes a **power-mean** composition `u_FPL(FQ)` that encodes the objective priorities. For Hopper:
   $$\phi_\text{hopper} = \Big(\textstyle\bigwedge^{p}\vec F_\text{speed}\Big)\ \wedge^{p}\ \Big(\textstyle\bigwedge^{p}\vec F_\text{action}\Big)$$
   where `∧ᵖ` is the power mean with `p ≤ 0` (paper uses `p ∈ {0, −1}`). Because `p ≤ 0` pulls the result toward the *smallest* fulfillment, an agent that stands still (`F_speed ≈ 0`) scores ≈ 0 — this is what kills the standing-still reward hack the paper reports.
3. **Observed-fulfillment regularizer.** The critic loss adds `α_FV · L_FV`, a supervised term against the Monte-Carlo *observed* fulfillment `FVᵒᵇˢ` (a conservative under-estimate) to counter overestimation bias — the paper's lighter-weight alternative to CrossQ/TQC/REDQ.
4. **Performance-based noise.** Exploration is parameter-space noise on the actor, `θ ~ N(θ, σ|θ|)`.

### Algorithm 1 (BPG)
```
Initialize networks and targets π, π_targ and FQ, FQ_targ;  replay buffer B
repeat
  Receive initial state s₁
  for each timestep t in episode:
      θ_π ← N(θ_π, σ·θ_π_prev)                      # performance-based noise
      aₜ ← π(sₜ);  execute aₜ;  store (rₜ, sₜ, aₜ, sₜ₊₁) in B
  Compute and store FVᵒᵇˢ for each step in B
  for each training iteration:
      (s, a, r, s', FVᵒᵇˢ) ~ B
      y_TD ← (1−γ)·r + γ·FQ_targ(s', π_targ(s'))
      L_TD ← μ̄₂( y_TD − FQ(s,a) )
      L_FV ← μ̄₂( FVᵒᵇˢ − FQ(s,a) )
      update critic with −∇(L_TD + α_FV·L_FV)
      J    ← μ̄₂( u_FPL( FQ(sₜ, π(sₜ)) ) )
      update actor with ∇_θπ J
  Update target networks
until convergence
```
`μ̄ₚ` is the power mean (`μ̄₂` = root-mean-square)."

In [ ]:
import numpy as np
import torch
import torch.nn as nn

DEVICE = "cpu"  # CPU torch build; Hopper-scale BPG trains fine here


# ======================================================================== #
# Fulfillment Priority Logic (FPL): power mean + the Hopper specification
# ======================================================================== #
def power_mean(x: torch.Tensor, p: float, dim: int = -1, eps: float = 1e-6):
    """Generalised power mean  (mean_i x_i^p)^(1/p)  over `dim`.

    p -> -inf : min,  p=-1 : harmonic,  p=0 : geometric,  p=1 : arithmetic.
    Inputs are clamped to [eps, 1] so p<=0 stays finite on zero fulfillments.
    """
    x = x.clamp(eps, 1.0)
    if abs(p) < 1e-8:                       # geometric mean (p -> 0)
        return torch.exp(x.log().mean(dim=dim))
    return x.pow(p).mean(dim=dim).pow(1.0 / p)


# --- Hopper-v5 fulfillment reward  R in [0,1]^N_OBJ  (the MF-MDP reward) ---
# Hopper-v5 observation layout: index 5 is the forward (x) velocity of the torso.
V_REF    = 2.0                       # forward speed [m/s] treated as fully fulfilled
N_SPEED  = 1                         # F_speed  components (forward velocity)
N_ACTION = 3                         # F_action components (one per joint torque)
N_OBJ    = N_SPEED + N_ACTION        # = 4


def hopper_fulfillments(next_obs: np.ndarray, action: np.ndarray) -> np.ndarray:
    """Vector reward R in [0,1]^N_OBJ for one transition.

      F_speed  : forward-velocity fulfillment (0 at rest/backwards, 1 at >=V_REF).
      F_action : 1 - |a_i| per joint -> fulfillment of minimising each torque.

    The paper's phi_hopper composes F_speed for "the velocity of each limb" and
    F_action for "minimising the three joint torques"; the exact signal->[0,1]
    maps are a design choice, kept here in one place so they are easy to retune.
    """
    vx = float(next_obs[5])
    f_speed  = np.clip(vx / V_REF, 0.0, 1.0)
    f_action = 1.0 - np.abs(np.asarray(action, dtype=np.float64))
    return np.concatenate([[f_speed], f_action]).astype(np.float32)


def u_fpl(fq: torch.Tensor, p: float) -> torch.Tensor:
    """u_FPL for phi_hopper = (AND^p F_speed) AND^p (AND^p F_action).

    fq: (..., N_OBJ) fulfillment Q-values -> scalar utility per row in [0,1].
    """
    speed  = power_mean(fq[..., :N_SPEED], p)            # AND^p F_speed
    action = power_mean(fq[..., N_SPEED:], p)            # AND^p F_action
    return power_mean(torch.stack([speed, action], dim=-1), p)


print("FPL ready | N_OBJ =", N_OBJ, "| torch", torch.__version__)

In [ ]:
# ======================================================================== #
# Networks: deterministic actor + vector Fulfillment-Q critic
# ======================================================================== #
def mlp(sizes, act=nn.ReLU, out_act=nn.Identity):
    layers = []
    for i in range(len(sizes) - 1):
        layers += [nn.Linear(sizes[i], sizes[i + 1]),
                   act() if i < len(sizes) - 2 else out_act()]
    return nn.Sequential(*layers)


class Actor(nn.Module):
    """Deterministic policy  pi(s) -> action in [-1, 1]  (tanh output)."""
    def __init__(self, obs_dim, act_dim, hidden=(256, 256)):
        super().__init__()
        self.net = mlp([obs_dim, *hidden, act_dim], out_act=nn.Tanh)

    def forward(self, s):
        return self.net(s)


class FQCritic(nn.Module):
    """Fulfillment Q-values  FQ(s,a) in [0,1]^N_OBJ  (sigmoid output keeps the
    FPL [0,1] semantics; consistent with y_TD = (1-g)r + g*FQ' which also in [0,1])."""
    def __init__(self, obs_dim, act_dim, n_obj, hidden=(256, 256)):
        super().__init__()
        self.net = mlp([obs_dim + act_dim, *hidden, n_obj])

    def forward(self, s, a):
        return torch.sigmoid(self.net(torch.cat([s, a], dim=-1)))

In [ ]:
# ======================================================================== #
# Replay buffer  (+ per-step observed fulfillment  FV^obs)
# ======================================================================== #
def compute_fv_obs(R, gamma, truncated):
    """FV^obs_t = (1-g) * sum_k g^k r_{t+k}  +  TRUNCATED * g^n * r_last.

    Normalised Monte-Carlo discounted return of the vector reward -> [0,1]^n.
    Coming from an older / exploratory policy it under-estimates fulfillment,
    so it pulls the critic down and counters overestimation bias.
    """
    R = np.asarray(R, np.float32)
    T = len(R)
    S = np.zeros_like(R)
    nxt = np.zeros(R.shape[1], np.float32)
    for t in range(T - 1, -1, -1):              # S_t = r_t + g * S_{t+1}
        S[t] = R[t] + gamma * nxt
        nxt = S[t]
    fv = (1.0 - gamma) * S
    if truncated:                               # geometric tail: last reward repeats forever
        rem = np.arange(T, 0, -1)               # remaining steps n at each t
        fv += (gamma ** rem)[:, None] * R[-1][None, :]
    return fv.astype(np.float32)


class ReplayBuffer:
    def __init__(self, cap, obs_dim, act_dim, n_obj):
        self.cap = cap
        self.s    = np.zeros((cap, obs_dim), np.float32)
        self.a    = np.zeros((cap, act_dim), np.float32)
        self.r    = np.zeros((cap, n_obj),  np.float32)
        self.s2   = np.zeros((cap, obs_dim), np.float32)
        self.term = np.zeros((cap, 1), np.float32)      # natural termination -> no bootstrap
        self.fv   = np.zeros((cap, n_obj), np.float32)  # FV^obs
        self.idx, self.full = 0, False

    def add_episode(self, S, A, R, S2, TERM, gamma, truncated):
        """Store a whole episode plus its per-step FV^obs (needs the full return)."""
        FV = compute_fv_obs(R, gamma, truncated)
        for i in range(len(S)):
            j = self.idx
            self.s[j], self.a[j], self.r[j]      = S[i], A[i], R[i]
            self.s2[j], self.term[j], self.fv[j] = S2[i], TERM[i], FV[i]
            self.idx = (self.idx + 1) % self.cap
            self.full = self.full or self.idx == 0

    def sample(self, n):
        hi = self.cap if self.full else self.idx
        k = np.random.randint(0, hi, size=n)
        t = lambda x: torch.as_tensor(x[k], device=DEVICE)
        return t(self.s), t(self.a), t(self.r), t(self.s2), t(self.term), t(self.fv)

    def __len__(self):
        return self.cap if self.full else self.idx

In [ ]:
# ======================================================================== #
# BPG agent  (Algorithm 1)
# ======================================================================== #
class BPG:
    def __init__(self, obs_dim, act_dim, *, gamma=0.9, tau=0.005, p=-1.0,
                 alpha_fv=0.75, actor_lr=1e-3, critic_lr=1e-3, hidden=(256, 256)):
        self.gamma, self.tau, self.p, self.alpha_fv = gamma, tau, p, alpha_fv
        self.obs_dim, self.act_dim, self.hidden = obs_dim, act_dim, hidden
        self.actor    = Actor(obs_dim, act_dim, hidden).to(DEVICE)
        self.actor_t  = Actor(obs_dim, act_dim, hidden).to(DEVICE)
        self.critic   = FQCritic(obs_dim, act_dim, N_OBJ, hidden).to(DEVICE)
        self.critic_t = FQCritic(obs_dim, act_dim, N_OBJ, hidden).to(DEVICE)
        self.actor_t.load_state_dict(self.actor.state_dict())
        self.critic_t.load_state_dict(self.critic.state_dict())
        self.a_opt = torch.optim.Adam(self.actor.parameters(),  lr=actor_lr)
        self.c_opt = torch.optim.Adam(self.critic.parameters(), lr=critic_lr)

    @torch.no_grad()
    def act(self, s, actor=None):
        actor = actor if actor is not None else self.actor
        s = torch.as_tensor(s, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        return actor(s).squeeze(0).cpu().numpy()

    def perturbed_actor(self, sigma):
        """Performance-based parameter-space noise:  theta ~ N(theta, sigma|theta|)."""
        noisy = Actor(self.obs_dim, self.act_dim, self.hidden).to(DEVICE)
        noisy.load_state_dict(self.actor.state_dict())
        with torch.no_grad():
            for prm in noisy.parameters():
                prm.add_(sigma * prm.abs() * torch.randn_like(prm))
        return noisy

    def train_step(self, buf, batch_size):
        s, a, r, s2, term, fv = buf.sample(batch_size)

        # --- critic: TD loss  +  observed-fulfillment regulariser ---
        with torch.no_grad():
            y = (1 - self.gamma) * r + self.gamma * (1 - term) * \
                self.critic_t(s2, self.actor_t(s2))            # y_TD, bootstrapped
        fq   = self.critic(s, a)
        l_td = torch.sqrt(((y  - fq) ** 2).mean())             # mu_2 residual vs TD target
        l_fv = torch.sqrt(((fv - fq) ** 2).mean())             # mu_2 residual vs FV^obs
        c_loss = l_td + self.alpha_fv * l_fv
        self.c_opt.zero_grad(); c_loss.backward(); self.c_opt.step()

        # --- actor: ascend the FPL utility of FQ(s, pi(s)) ---
        u = u_fpl(self.critic(s, self.actor(s)), self.p)       # J = mu_2(u_FPL)
        a_loss = -torch.sqrt((u ** 2).mean())
        self.a_opt.zero_grad(); a_loss.backward(); self.a_opt.step()

        # --- Polyak target update ---
        with torch.no_grad():
            for tp, sp in zip(self.actor_t.parameters(),  self.actor.parameters()):
                tp.mul_(1 - self.tau).add_(self.tau * sp)
            for tp, sp in zip(self.critic_t.parameters(), self.critic.parameters()):
                tp.mul_(1 - self.tau).add_(self.tau * sp)
        return (float(l_td.detach()), float(l_fv.detach()), float(u.mean().detach()))

In [ ]:
# ======================================================================== #
# Training loop  (episode collection -> FV^obs -> updates -> target sync)
# ======================================================================== #
import gymnasium as gym


def evaluate(agent, episodes=5, seed=10_000):
    """Greedy (noise-free) rollouts; returns the mean fulfillment value:
    hopper_fulfillments averaged over objectives, steps and episodes."""
    env = gym.make("Hopper-v5")
    fvs = []
    for e in range(episodes):
        s, _ = env.reset(seed=seed + e)
        done = False
        while not done:
            a = agent.act(s)
            s, r, term, trunc, _ = env.step(a)
            fvs.append(hopper_fulfillments(s, a).mean())
            done = term or trunc
    env.close()
    return float(np.mean(fvs))


def train(target_fv=0.8, *, max_steps=200_000, seed=0, sigma=0.1, batch_size=256,
          warmup=1000, updates_per_step=1, eval_every=2000, log=print):
    """Run BPG until the greedy-eval mean fulfillment reaches `target_fv`
    (checked every `eval_every` steps), with `max_steps` as a hard cap."""
    np.random.seed(seed); torch.manual_seed(seed)
    env = gym.make("Hopper-v5")
    obs_dim, act_dim = env.observation_space.shape[0], env.action_space.shape[0]
    agent = BPG(obs_dim, act_dim)
    buf = ReplayBuffer(200_000, obs_dim, act_dim, N_OBJ)

    steps, next_eval, reached = 0, eval_every, False
    hist = {"step": [], "behavior_fv": [], "eval_step": [], "eval_fv": []}
    while steps < max_steps and not reached:
        s, _ = env.reset(seed=seed + 1000 + len(hist["behavior_fv"]))
        noisy = agent.perturbed_actor(sigma)        # performance-based noise, fixed per episode
        S, A, R, S2, TERM = [], [], [], [], []
        done, truncated = False, False
        while not done:
            a = env.action_space.sample() if steps < warmup else agent.act(s, noisy)
            s2, rew, terminated, truncated, _ = env.step(a)
            S.append(s); A.append(a); R.append(hopper_fulfillments(s2, a))
            S2.append(s2); TERM.append([float(terminated)])
            s = s2; steps += 1
            done = terminated or truncated
            if len(buf) > batch_size and steps >= warmup:
                for _ in range(updates_per_step):
                    agent.train_step(buf, batch_size)
            if steps >= next_eval:
                fv = evaluate(agent)
                hist["eval_step"].append(steps); hist["eval_fv"].append(fv)
                log(f"  [eval] step {steps:6d}  mean fulfillment {fv:.3f}")
                next_eval += eval_every
                if fv >= target_fv:
                    reached = True
                    done = truncated = True     # cut the episode off here
        # store the whole episode together with its per-step FV^obs
        buf.add_episode(np.array(S, np.float32),  np.array(A, np.float32),
                        np.array(R, np.float32),  np.array(S2, np.float32),
                        np.array(TERM, np.float32), agent.gamma, truncated)
        hist["step"].append(steps)
        hist["behavior_fv"].append(float(np.mean(R)))
        if len(hist["behavior_fv"]) % 10 == 0:
            log(f"step {steps:6d} | ep {len(hist['behavior_fv']):4d} | "
                f"behavior mean fulfillment {np.mean(hist['behavior_fv'][-10:]):.3f}")
    env.close()
    log(f"target mean fulfillment {target_fv} reached at step {steps}" if reached
        else f"stopped at max_steps={max_steps} without reaching {target_fv}")
    return agent, hist


In [ ]:
# Train until the greedy-eval mean fulfillment reaches 0.8 (hard cap: max_steps).
# Evaluation runs every eval_every steps; expect this to take a while on CPU.
agent, hist = train(target_fv=0.8, max_steps=200_000, seed=0)
print("final greedy eval (mean fulfillment):", evaluate(agent, episodes=10))


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(hist["step"], hist["behavior_fv"], lw=1, alpha=0.6, label="behavior episodes")
if hist["eval_step"]:
    ax.plot(hist["eval_step"], hist["eval_fv"], "-o", color="tab:red", label="greedy eval")
ax.axhline(0.8, color="gray", ls="--", lw=1, label="target 0.8")
ax.set(title="Mean fulfillment value", xlabel="env steps", ylabel="mean fulfillment [0,1]")
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()


## Live run — watch the learned policy

Render the greedy (noise-free) trained policy on `Hopper-v5` and show it inline as a video with **at least 20 seconds of footage** — the env is reset and re-rolled until enough frames are collected. Requires a MuJoCo GL backend; `MUJOCO_GL=egl` works headless. Run the training cell first so `agent` is defined.


In [ ]:
# ======================================================================== #
# Live run: watch the learned greedy policy hop  (inline video, >= 20 s)
# ======================================================================== #
# MUJOCO_GL must be set before the FIRST MuJoCo render in this kernel.
#   'egl'  -> headless-safe offscreen GL (works without a display)
#   'glfw' -> use instead if you have a display and want a pop-up window
import os
os.environ.setdefault("MUJOCO_GL", "egl")

import imageio.v2 as imageio
from pathlib import Path
from IPython.display import Video


def rollout_video(agent, path="runs/hopper_bpg.mp4", seed=12345, fps=30, min_seconds=20):
    """Roll out the greedy (noise-free) policy, resetting on termination, until
    at least `min_seconds` of footage is captured; encode it to mp4."""
    env = gym.make("Hopper-v5", render_mode="rgb_array")
    frames, fvs, ep = [], [], 0
    while len(frames) < fps * min_seconds:
        s, _ = env.reset(seed=seed + ep)
        ep += 1
        frames.append(env.render())
        done = False
        while not done:
            a = agent.act(s)                    # deterministic pi(s)
            s, r, term, trunc, _ = env.step(a)
            frames.append(env.render())
            fvs.append(hopper_fulfillments(s, a).mean())
            done = term or trunc
    env.close()
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    imageio.mimwrite(path, frames, fps=fps, codec="libx264")
    print(f"{len(frames) / fps:.1f} s of footage ({ep} episode(s), {len(frames)} frames) | "
          f"mean fulfillment {np.mean(fvs):.3f}  ->  {path}")
    return path


video_path = rollout_video(agent)         # `agent` comes from the training cell above
Video(video_path, embed=True, width=480)  # embedded so it renders even on a remote kernel


### Notes & how to reach the paper's numbers

- **Fulfillment everywhere.** Training optimizes the FPL utility of the *vector* fulfillment reward (`hopper_fulfillments`), and evaluation reports the **mean fulfillment value** — `hopper_fulfillments` averaged over objectives, steps and eval episodes. The env's native scalar reward is not used as a metric anywhere. (The paper instead evaluates with the benchmark's scalar reward; this notebook deliberately drops it.)
- **Stopping rule.** `train(target_fv=0.8)` keeps collecting and updating until the greedy-eval mean fulfillment reaches **0.8** (checked every `eval_every` steps), with `max_steps` as a hard cap.
- **Knobs that matter** (`BPG(...)` / `train(...)`):
  - `gamma` — discount of the fulfillment Q-values (this notebook uses `0.9`).
  - `p` — conjunction strictness of `∧ᵖ` (paper uses `0` or `−1`; more negative ⇒ stronger "*every* objective must be satisfied" guarantee).
  - `alpha_fv` — weight of the `FVᵒᵇˢ` overestimation regularizer (paper: `0.75`).
  - `sigma` — performance-based parameter-noise scale.
  - `V_REF`, `hopper_fulfillments`, `u_fpl` — **the FPL spec itself**; edit these to change the *intended behavior*. That is the paper's whole point: behavior follows the declarative spec, not hand-tuned linear reward weights.


# Part 1 — Verification: the algorithm above **is** BPG (Algorithm 1)

Line-by-line mapping between the paper (El Mabsout et al., IROS 2025) and the code
in this notebook. Nothing here is REINFORCE or PPO — there is no log-probability,
no importance ratio, no clipping, no advantage estimate. It is DDPG restructured
around FPL, exactly as Algorithm 1 specifies:

| Algorithm 1 / paper equation | Notebook code |
|---|---|
| Vector critic of **Fulfillment Q-values** `FQ = (1−γ)Q ∈ [0,1]ⁿ` (Def III-D, Eq 2) | `FQCritic` — sigmoid head, `N_OBJ` outputs |
| `y_TD = (1−γ)·r + γ·FQ_targ(s′, π_targ(s′))` | `BPG.train_step`: `y = (1-γ)*r + γ*(1-term)*critic_t(s2, actor_t(s2))` |
| `L_TD = μ̄₂(y_TD − FQ)` , `L_FV = μ̄₂(FVᵒᵇˢ − FQ)` , critic ← `−∇(L_TD + α_FV·L_FV)` (§V-A) | `l_td + self.alpha_fv * l_fv` with RMS (μ̄₂) residuals, `alpha_fv=0.75` (paper's value) |
| `FVᵒᵇˢ = (1−γ)Σγᵗr⃗ₜ (+ truncation tail)` — conservative MC target (§V-A) | `compute_fv_obs` inside `ReplayBuffer.add_episode` |
| Actor ascends `J = μ̄₂(u_FPL(FQ(s, π(s))))` | `a_loss = -torch.sqrt((u ** 2).mean())` with `u = u_fpl(critic(s, actor(s)), p)` |
| `u_FPL` = power-mean conjunction of the FPL spec, `p ≤ 0` (Eqs 3–7) | `u_fpl` = nested `power_mean(·, p=-1)` encoding `φ_hopper = (∧ᵖF_speed) ∧ᵖ (∧ᵖF_action)` |
| Performance-based noise `θ ~ N(θ, σ·θ_prev)` per episode | `perturbed_actor(sigma)`, fixed for the episode |
| Polyak target sync | `tp.mul_(1-τ).add_(τ·sp)`, `τ=0.005` |
| Off-policy replay buffer `B` | `ReplayBuffer`, per-episode insertion (FVᵒᵇˢ needs the full return) |

The next cell runs three mechanical checks: the torch `power_mean` matches the
`analytic_mppi` numpy implementation (so the RL side and the MPPI side aggregate
fulfillments identically), the TD target respects the `[0,1]` invariant that makes
the sigmoid critic head consistent, and `u_fpl` behaves like an FPL conjunction
(range, unit element, monotonicity).

In [ ]:
# === Verification checks ========================================================
from analytic_mppi.tasks.base import power_mean as np_power_mean

rng_chk = np.random.default_rng(0)

# (1) torch power_mean == analytic_mppi numpy power_mean (same eps) on [0,1] inputs
x = rng_chk.uniform(0.0, 1.0, size=(64, 4))
for p_chk in (-2.0, -1.0, 0.0, 0.5, 1.0):
    a = power_mean(torch.as_tensor(x), p_chk).numpy()
    b = np_power_mean(x, p_chk, eps=1e-6)          # match the torch clamp floor
    assert np.allclose(a, b, atol=1e-6), (p_chk, np.abs(a - b).max())
print("PASS (1) torch power_mean == analytic_mppi.tasks.base.power_mean "
      "for p in {-2,-1,0,0.5,1}")

# (2) TD-target [0,1] invariant: y = (1-γ)r + γ(1-term)·FQ stays in [0,1]
#     whenever r, FQ in [0,1] — the property that justifies the sigmoid critic head
g = 0.9
r = torch.rand(1000, 4); fq = torch.rand(1000, 4); term = (torch.rand(1000, 1) < 0.3).float()
y = (1 - g) * r + g * (1 - term) * fq
assert y.min() >= 0.0 and y.max() <= 1.0
print(f"PASS (2) y_TD in [0,1]:  min {y.min():.4f}  max {y.max():.4f}")

# (3) u_fpl is an FPL conjunction: range [0,1], u(1,...,1)=1, monotone per argument
u_ones = float(u_fpl(torch.ones(1, N_OBJ), -1.0))
assert abs(u_ones - 1.0) < 1e-5
v = torch.full((1, N_OBJ), 0.5)
u_lo = float(u_fpl(v, -1.0))
v2 = v.clone(); v2[0, 2] = 0.9
u_hi = float(u_fpl(v2, -1.0))
assert 0.0 <= u_lo <= u_hi <= 1.0
print(f"PASS (3) u_fpl: u(1)= {u_ones:.5f}, monotone ({u_lo:.3f} -> {u_hi:.3f} "
      "after raising one term)")

print("\nall BPG verification checks passed")

# Part 2 — Teacher–student: MPPI-FPL teacher → BPG student

**Setup.** Both teacher and student live in the *repo's* hopper
(`analytic_mppi.tasks.HopperTask`, `envs/hopper/scene.xml`, dt = 0.02) and share its
four FPL fulfillment terms `running_cost_terms_f ∈ [0,1]⁴`:
**height** `exp(−((z−1.0)/0.1)²)`, **orientation** `(z_axis+1)/2`,
**velocity** `exp(−((v−0.1)/0.5)²)`, **control** `1−mean(u²)`.
The Gym Hopper-v5 demo above is untouched; this part is self-contained.
(The env wrapper and drivers below are written generically — Part 3 reuses them
unchanged on `walker` and `g1_standup`.)

- **Teacher** = the package's `MPPIv2` with **FPL cost terms**
  (`use_fpl_cost=True`) — it scores every sampled rollout by the discounted
  power-mean of the fulfillment terms and picks the best sample's knots.
- **Student** = the **BPG agent verified in Part 1**, with the FPL utility swapped
  to the flat conjunction `u = power_mean(terms, p)` so teacher objective ≡
  student objective ≡ evaluation metric.

**Two empirically-forced choices** (probed before training):
1. **Aggregation `p = 0.1`** (the repo's MPPI default `fpl_p`), *not* the paper's
   `p = −1`: holding the required crouch (torso z ≈ 1.0, from a 1.25 standing pose)
   costs sustained torque, capping the control term at ≈ 0.78 — under the harmonic
   mean even a strong MPPI teacher tops out at episode-mean ≈ 0.80, i.e. the 0.8
   success threshold *is* the task ceiling. At `p = 0.1` the same behaviour scores
   ≈ 0.82–0.85, so "steps to 0.8" is a measurable target.
2. **Strengthened hopper teacher** (`num_samples=256, iterations=2,
   noise_level=0.2, plan_horizon=1.0`, cubic/15 knots): the tuned fast config
   (K=120, h=0.6) oscillates and sinks over 400-step episodes (mean ≈ 0.65–0.72),
   which would give the student a weak demonstrator.

**Protocol**:

| Phase | What happens | Env steps counted |
|---|---|---|
| T — demonstrate | teacher collects ≈ 4,800 steps of episodes → student replay buffer (with FVᵒᵇˢ) | yes |
| O — distill | offline BPG updates **+ behaviour-cloning term** on the actor (DDPGfD-style), with **early stopping** (< 0.01 gain over 3 evals, ≤ 20k updates) and **best-checkpoint restore** — the student exits at its peak (see Part 3) | **no** (0 steps) |
| S — self-train | BC dropped; standard **online BPG** (parameter noise, own rollouts, demos retained in buffer) until eval ≥ 0.8 or the 60k-step budget | yes |

**Baseline**: BPG-only from scratch — same env, seed, eval cadence and budget.

**Metric**: eval fulfillment = mean over 5 greedy episodes (fixed eval seeds) of
`sum_t power_mean(r⃗_t, p) / HORIZON` on realized transitions — never the critic's
estimate. Normalising by the full horizon means a terminated (fallen) episode
scores 0 for its missing steps; probing showed the per-surviving-step average is
degenerate on g1_standup (falling after 2 s still averaged ~0.92).
**steps-to-0.8** = first *total* env-step count (teacher + student) at which it
crosses 0.8. Episodes: 400-step truncation, per-env fall termination,
gym-style ±5·10⁻³ reset noise.

In [ ]:
# === Experiment config + the shared environment wrapper =========================
import time
import mujoco
from analytic_mppi.tasks import make_task
from analytic_mppi.tasks.base import power_mean as np_power_mean
from analytic_mppi.dynamics import MujocoBackend
from analytic_mppi.controllers.mppi_v2 import MPPIv2

SEED          = 0
FPL_P         = 0.1        # repo MPPI default fpl_p; see Part-2 markdown for why not -1
GAMMA         = 0.9
TARGET_F      = 0.8        # success threshold on eval fulfillment
TOTAL_BUDGET  = 60_000     # total env steps per run (teacher demos + online)
TEACHER_STEPS = 4_800      # ~12 episodes of demonstrations (8% of budget)
HORIZON       = 400        # episode truncation
RESET_NOISE   = 5e-3       # gym-style uniform reset noise on qpos/qvel

# Per-env facts (probed before training, under the horizon-normalized metric):
#  - hopper:  planar, rootx=qpos[0]. Strengthened teacher holds the z~1.0 crouch;
#             healthy_z must be 0.5 (NOT the Hopper-v5-style 0.7): the crouch sits
#             at z~1.0 and recoverable wobbles dip below 0.7 -- with a 0.7 floor
#             the teacher "dies" mid-correction at ~150 steps (0.44 fulfillment),
#             with 0.5 it recovers and scores 0.83 over full 400-step episodes.
#  - walker:  planar, root order z-then-x -> rootx=qpos[1]; must WALK at 1.5 m/s.
#             K=256/iter2/noise .3/cubic-8 sustains 0.81; the tuned fast config and
#             a zero-spline variant plateau at 0.74-0.76 (survive but track worse).
#  - g1_standup: free-joint humanoid, drop root x,y = qpos[0:2]; run.py convention
#             starts from the "stand" keyframe; 3 fulfillment terms
#             (orient/height/nominal). The G1 stands PASSIVELY at zero torque, so
#             the fulfillment metric must horizon-normalize (see episode_fulfillment)
#             or falling policies score ~0.92. K=256/iter2 teacher: 0.87 (~310 ms/plan).
ENV_CFG = {
    "hopper": dict(
        drop_qpos=(0,), healthy_z=0.5, init="zeros",
        teacher=dict(num_samples=256, noise_level=0.2, plan_horizon=1.0,
                     num_knots=15, spline_type="cubic", iterations=2)),
    "walker": dict(
        drop_qpos=(1,), healthy_z=0.8, init="zeros",
        teacher=dict(num_samples=256, noise_level=0.3, plan_horizon=0.6,
                     num_knots=8, spline_type="cubic", iterations=2)),
    "g1_standup": dict(
        drop_qpos=(0, 1), healthy_z=0.5, init="keyframe:stand",
        teacher=dict(num_samples=256, noise_level=0.2, plan_horizon=0.6,
                     num_knots=8, spline_type="cubic", iterations=2)),
}


class FPLEnv:
    """Gym-style wrapper: repo task + MujocoBackend, vector fulfillment reward.

    obs = [qpos minus the root-translation coords, qvel]
    reward = task.running_cost_terms_f in [0,1]^n_obj
    One backend serves closed-loop stepping (backend.data) AND the MPPI teacher's
    batched planning rollouts (separate _thread_data) without interference.
    """

    def __init__(self, name, seed=SEED):
        cfg = ENV_CFG[name]
        self.name = name
        self.task = make_task(name)
        self.backend = MujocoBackend(self.task.model_path)
        self.keep_qpos = np.array(
            [i for i in range(self.backend.nq) if i not in cfg["drop_qpos"]])
        self.obs_dim = len(self.keep_qpos) + self.backend.nv
        self.act_dim = self.backend.nu
        self.healthy_z = cfg["healthy_z"]
        self.rng = np.random.default_rng(seed)
        self.t = 0
        # base reset state: zeros (== default pose) or a named keyframe
        base = np.zeros(self.backend.nstate)
        if cfg["init"].startswith("keyframe:"):
            kf = self.backend.model.keyframe(cfg["init"].split(":", 1)[1])
            base[self.backend.qpos_slice] = kf.qpos
        self._base_state = base
        self.reset()
        self.n_obj = int(self._r_probe().shape[-1])

    def _obs(self):
        d = self.backend.data
        return np.concatenate([np.asarray(d.qpos)[self.keep_qpos],
                               d.qvel]).astype(np.float32)

    def _r_probe(self):
        d = self.backend.data
        return self.task.running_cost_terms_f(
            d.qpos, d.qvel, d.sensordata, np.zeros(self.act_dim))

    def torso_z(self):
        return float(self.task._torso_height(self.backend.data.sensordata))

    def reset(self):
        state = self._base_state.copy()
        state[self.backend.qpos_slice] += self.rng.uniform(
            -RESET_NOISE, RESET_NOISE, self.backend.nq)
        state[self.backend.qvel_slice] += self.rng.uniform(
            -RESET_NOISE, RESET_NOISE, self.backend.nv)
        self.backend.set_state(state)               # set_state runs mj_forward
        self.t = 0
        return self._obs()

    def step(self, u):
        u = np.clip(np.asarray(u, dtype=np.float64),
                    self.task.u_min, self.task.u_max)
        self.backend.step(u)
        # mj_step leaves data.sensordata at the PRE-step state; refresh before reading
        mujoco.mj_forward(self.backend.model, self.backend.data)
        d = self.backend.data
        r_vec = self.task.running_cost_terms_f(d.qpos, d.qvel, d.sensordata, u)
        self.t += 1
        terminated = self.torso_z() < self.healthy_z
        truncated = (not terminated) and (self.t >= HORIZON)
        return self._obs(), r_vec.astype(np.float32), terminated, truncated

    def full_state(self):
        """FULLPHYSICS state of the wrapper's exact current state (for the teacher)."""
        return self.backend.get_state()


for _n in ENV_CFG:
    _e = FPLEnv(_n)
    _e.reset()
    print(f"{_n:11s} obs_dim={_e.obs_dim:3d} act_dim={_e.act_dim:3d} "
          f"n_obj={_e.n_obj} dt={_e.backend.dt}  reset torso z={_e.torso_z():.3f}")

In [ ]:
# === Flat FPL utility + the fulfillment evaluation metric =======================
def u_fpl_flat(fq: torch.Tensor, p: float = FPL_P) -> torch.Tensor:
    """phi = AND^p over the task's fulfillment terms (flat conjunction).
    Same aggregation MPPIv2's _score_fpl applies per step (parity checked in Part 1);
    replaces the nested Hopper-v5 u_fpl for this experiment."""
    return power_mean(fq, p)


def episode_fulfillment(R, horizon=None):
    """(T, n_obj) per-step fulfillments -> sum_t power_mean(r_t, p) / HORIZON.

    Normalising by the FULL horizon (not the survived length T) makes a fallen
    (terminated) episode score 0 for every missing step. Without this, g1_standup
    is degenerate: a policy that stands 2 s and then falls averages ~0.92 over its
    surviving steps -- indistinguishable from standing the whole 8 s (the G1 is
    passively stable at zero torque, so surviving-step averages are always high)."""
    horizon = HORIZON if horizon is None else horizon
    return float(np.sum(np_power_mean(np.asarray(R), FPL_P, eps=1e-6)) / horizon)


def evaluate_fulfillment(policy_fn, env_name, episodes=5, eval_seed=10_000):
    """Greedy rollouts on fresh, deterministically-seeded envs.
    Returns (mean episode fulfillment, per-term means (n_obj,))."""
    scores, term_means = [], []
    for e in range(episodes):
        env = FPLEnv(env_name, seed=eval_seed + e)
        s, R = env.reset(), []
        done = False
        while not done:
            s, r_vec, term, trunc = env.step(policy_fn(s))
            R.append(r_vec)
            done = term or trunc
        scores.append(episode_fulfillment(R))
        term_means.append(np.mean(R, axis=0))
    return float(np.mean(scores)), np.mean(term_means, axis=0)


# --- smoke test: one random-action episode on the hopper ------------------------
_env = FPLEnv("hopper", seed=SEED)
_rng_smoke = np.random.default_rng(1)
s = _env.reset()
assert s.shape == (11,)
T_smoke, done = 0, False
while not done:
    s, r_vec, term, trunc = _env.step(_rng_smoke.uniform(-1, 1, 3))
    assert r_vec.shape == (4,) and (r_vec >= 0).all() and (r_vec <= 1).all()
    T_smoke += 1
    done = term or trunc
print(f"smoke test: random-action episode ended after {T_smoke} steps "
      f"({'fell' if term else 'truncated'}), final torso z = {_env.torso_z():.3f}")

In [ ]:
# === Teacher: MPPIv2 with FPL cost terms + reachability gate ====================
def make_teacher(env, seed=SEED):
    """Per-env MPPI config from ENV_CFG (each probed to sustain >~0.8 fulfillment)."""
    return MPPIv2(env.task, env.backend, temperature=0.05, seed=seed,
                  use_fpl_cost=True, fpl_p=FPL_P, fpl_gamma=GAMMA,
                  **ENV_CFG[env.name]["teacher"])


def run_teacher_episode(env, ctrl):
    """One closed-loop teacher episode. Returns (S, A, R, S2, TERM), truncated."""
    s = env.reset()
    ctrl.reset()                                   # zero the warm-start mean
    S, A, R, S2, TERM = [], [], [], [], []
    done = truncated = False
    while not done:
        u = np.clip(ctrl.act(env.full_state()),
                    env.task.u_min, env.task.u_max).astype(np.float32)
        s2, r_vec, term, truncated = env.step(u)
        S.append(s); A.append(u); R.append(r_vec); S2.append(s2)
        TERM.append([float(term)])
        s = s2
        done = term or truncated
    return (np.array(S, np.float32), np.array(A, np.float32), np.array(R, np.float32),
            np.array(S2, np.float32), np.array(TERM, np.float32)), truncated


def teacher_gate(env_name, episodes=3, seed=SEED + 100, log=print):
    """Reachability gate: the teacher's own fulfillment, measured BEFORE training."""
    env = FPLEnv(env_name, seed=seed)
    ctrl = make_teacher(env, seed=seed)
    t0, scores, terms = time.perf_counter(), [], []
    for _ in range(episodes):
        (_, _, R, _, _), _tr = run_teacher_episode(env, ctrl)
        scores.append(episode_fulfillment(R))
        terms.append(np.mean(R, axis=0))
    f = float(np.mean(scores))
    log(f"[{env_name}] teacher eval: {episodes} episodes in "
        f"{time.perf_counter()-t0:.0f}s | fulfillments {np.round(scores, 3)}")
    log(f"[{env_name}] teacher mean fulfillment = {f:.3f}   per-term = "
        f"{np.round(np.mean(terms, axis=0), 3)}")
    if f >= TARGET_F:
        log(f"[{env_name}] teacher >= {TARGET_F} -> target REACHABLE by imitation + RL")
    else:
        log(f"[{env_name}] WARNING: teacher < {TARGET_F} -> treat runs as diagnostic")
    return f


TEACHER_FS = {"hopper": teacher_gate("hopper")}

In [ ]:
# === Phase T: teacher demonstrations -> student replay buffer ===================
def collect_demos(env_name, n_steps=TEACHER_STEPS, seed=SEED, log=print):
    """Run whole teacher episodes until n_steps env steps are collected.
    Returns (buffer, actual_steps, per-episode fulfillments)."""
    env = FPLEnv(env_name, seed=seed)
    ctrl = make_teacher(env, seed=seed)
    buf = ReplayBuffer(200_000, env.obs_dim, env.act_dim, env.n_obj)
    steps, scores = 0, []
    t0 = time.perf_counter()
    while steps < n_steps:
        (S, A, R, S2, TERM), truncated = run_teacher_episode(env, ctrl)
        buf.add_episode(S, A, R, S2, TERM, GAMMA, truncated)
        steps += len(S)
        scores.append(episode_fulfillment(R))
    log(f"[{env_name}] teacher demos: {steps} env steps, {len(scores)} episodes, "
        f"buffer size {len(buf)}  ({time.perf_counter()-t0:.0f}s)")
    log(f"[{env_name}] demo episode fulfillments: mean {np.mean(scores):.3f}  "
        f"min {np.min(scores):.3f}  max {np.max(scores):.3f}")
    return buf, steps, scores


student_buf, teacher_steps, demo_scores = collect_demos("hopper")

In [ ]:
# === BPG + behaviour cloning, checkpointing, and early-stopped distillation =====
import copy


class BPGBC(BPG):
    """BPG whose actor loss can carry a behaviour-cloning term, whose FPL utility
    is the flat conjunction u_fpl_flat (the teacher's aggregation), and whose
    critic width follows the env's n_obj (g1_standup has 3 terms, not 4).
    With bc_coef=0 this is exactly BPG on the new spec."""

    def __init__(self, obs_dim, act_dim, *, n_obj=N_OBJ, critic_lr=1e-3, **kw):
        super().__init__(obs_dim, act_dim, critic_lr=critic_lr, **kw)
        if n_obj != N_OBJ:                      # rebuild the vector critic head
            self.critic = FQCritic(obs_dim, act_dim, n_obj, self.hidden).to(DEVICE)
            self.critic_t = FQCritic(obs_dim, act_dim, n_obj, self.hidden).to(DEVICE)
            self.critic_t.load_state_dict(self.critic.state_dict())
            self.c_opt = torch.optim.Adam(self.critic.parameters(), lr=critic_lr)

    # --- student checkpointing (all four networks) ---
    def snapshot(self):
        return {k: copy.deepcopy(getattr(self, k).state_dict())
                for k in ("actor", "actor_t", "critic", "critic_t")}

    def restore(self, ckpt):
        for k, sd in ckpt.items():
            getattr(self, k).load_state_dict(sd)

    def train_step(self, buf, batch_size, bc_coef=0.0):
        s, a, r, s2, term, fv = buf.sample(batch_size)

        # --- critic: identical to BPG.train_step ---
        with torch.no_grad():
            y = (1 - self.gamma) * r + self.gamma * (1 - term) * \
                self.critic_t(s2, self.actor_t(s2))
        fq = self.critic(s, a)
        l_td = torch.sqrt(((y - fq) ** 2).mean())
        l_fv = torch.sqrt(((fv - fq) ** 2).mean())
        c_loss = l_td + self.alpha_fv * l_fv
        self.c_opt.zero_grad(); c_loss.backward(); self.c_opt.step()

        # --- actor: flat FPL utility (+ optional BC toward the batch actions) ---
        pi = self.actor(s)
        u = u_fpl_flat(self.critic(s, pi), self.p)
        a_loss = -torch.sqrt((u ** 2).mean())
        if bc_coef > 0.0:
            a_loss = a_loss + bc_coef * ((pi - a) ** 2).mean()
        self.a_opt.zero_grad(); a_loss.backward(); self.a_opt.step()

        # --- Polyak target update (unchanged) ---
        with torch.no_grad():
            for tp, sp in zip(self.actor_t.parameters(), self.actor.parameters()):
                tp.mul_(1 - self.tau).add_(self.tau * sp)
            for tp, sp in zip(self.critic_t.parameters(), self.critic.parameters()):
                tp.mul_(1 - self.tau).add_(self.tau * sp)
        return (float(l_td.detach()), float(l_fv.detach()), float(u.mean().detach()))


def offline_train(agent, buf, env_name, *, max_updates=20_000, batch_size=256,
                  eval_every=2_000, bc_coef=1.0,
                  plateau_delta=0.01, plateau_patience=3, log=print):
    """Phase O: distill from the demo buffer with EARLY STOPPING + BEST-CHECKPOINT
    RESTORE. Offline actor-critic training on a fixed dataset overtrains (the
    first run peaked at 0.849 after 2k updates and decayed to 0.700 by 8k), so:
      - a full student checkpoint (actor/critic + targets) is saved at every eval
        that improves on the best fulfillment so far;
      - training stops early when improvement < plateau_delta for
        plateau_patience consecutive evals (or at max_updates);
      - the BEST checkpoint is restored before the online phase, so the student
        exits distillation at its peak, not at wherever the plateau rule fired.
    Costs 0 env steps. Returns (evals, best_f, best_upd)."""
    evals, best, best_upd, stall, ckpt = [], -np.inf, 0, 0, None
    for upd in range(1, max_updates + 1):
        agent.train_step(buf, batch_size, bc_coef=bc_coef)
        if upd % eval_every == 0:
            f, per_term = evaluate_fulfillment(lambda s: agent.act(s), env_name)
            evals.append((upd, f))
            log(f"  [offline] upd {upd:6d}  eval fulfillment {f:.3f}  "
                f"terms {np.round(per_term, 3)}")
            stall = 0 if f >= best + plateau_delta else stall + 1   # vs previous best
            if f > best:
                best, best_upd, ckpt = f, upd, agent.snapshot()
            if stall >= plateau_patience:
                log(f"  early stop: < {plateau_delta} improvement over "
                    f"{plateau_patience} evals -> student has learned what it can "
                    "from the teacher")
                break
    if ckpt is not None:
        agent.restore(ckpt)
        log(f"  restored best checkpoint: upd {best_upd}, eval fulfillment {best:.3f}")
    return evals, best, best_upd

In [ ]:
# === Phase S / baseline: online BPG driver ======================================
def online_train(agent, buf, env_name, *, step_offset=0, budget=TOTAL_BUDGET,
                 seed=SEED, sigma=0.1, batch_size=256, warmup=0,
                 updates_per_step=1, eval_every=1_000, target=TARGET_F, log=print):
    """Online BPG (bc_coef=0). History is keyed by TOTAL env steps = step_offset +
    online steps, so teacher demonstrations count toward steps-to-target.
    Stops at eval fulfillment >= target or when the total budget is exhausted."""
    np.random.seed(seed); torch.manual_seed(seed)
    env = FPLEnv(env_name, seed=seed + 500)
    hist = {"eval_step": [], "eval_f": [], "eval_terms": []}

    def do_eval(total):
        f, terms = evaluate_fulfillment(lambda s_: agent.act(s_), env_name)
        hist["eval_step"].append(total)
        hist["eval_f"].append(f)
        hist["eval_terms"].append(terms)
        log(f"  [online] total steps {total:6d}  eval fulfillment {f:.3f}")
        return f

    reached = do_eval(step_offset) >= target      # credit "ready out of distillation"
    steps, next_eval = 0, eval_every
    remaining = budget - step_offset
    while not reached and steps < remaining:
        s = env.reset()
        noisy = agent.perturbed_actor(sigma)
        S, A, R, S2, TERM = [], [], [], [], []
        terminated = truncated = False
        while True:
            a = (np.random.uniform(-1, 1, env.act_dim) if steps < warmup
                 else agent.act(s, noisy))
            s2, r_vec, terminated, truncated = env.step(a)
            S.append(s); A.append(np.asarray(a, np.float32)); R.append(r_vec)
            S2.append(s2); TERM.append([float(terminated)])
            s = s2; steps += 1
            if len(buf) > batch_size and steps >= warmup:
                for _ in range(updates_per_step):
                    agent.train_step(buf, batch_size)
            if steps >= next_eval:
                next_eval += eval_every
                if do_eval(step_offset + steps) >= target:
                    reached = True
            if terminated or truncated or reached or steps >= remaining:
                break
        # episodes cut by target/budget are stored as truncated for the FV^obs tail
        buf.add_episode(np.array(S, np.float32), np.array(A, np.float32),
                        np.array(R, np.float32), np.array(S2, np.float32),
                        np.array(TERM, np.float32), agent.gamma,
                        truncated or not (terminated or truncated))
    return hist, steps


def steps_to_target(hist, target=TARGET_F):
    """First total-step count at which eval fulfillment >= target, else None."""
    for st, f in zip(hist["eval_step"], hist["eval_f"]):
        if f >= target:
            return st
    return None


EXPERIMENTS = {}   # env_name -> dict(ts=..., bl=...) filled by the run cells below

In [ ]:
# === Run A (hopper): teacher-student = demonstrate -> distill -> online BPG =====
np.random.seed(SEED); torch.manual_seed(SEED)
agent_ts = BPGBC(11, 3, n_obj=4)

print(f"phase O: offline distillation on {len(student_buf)} teacher transitions")
t0 = time.perf_counter()
offline_evals, best_off_f, best_off_upd = offline_train(
    agent_ts, student_buf, "hopper", bc_coef=1.0)
offline_updates = offline_evals[-1][0] if offline_evals else 0
print(f"offline phase: {offline_updates} updates in {time.perf_counter()-t0:.0f}s\n")

print(f"phase S: online BPG from total step {teacher_steps} (warmup=0 — the buffer "
      "already holds on-task demos)")
t0 = time.perf_counter()
hist_ts, online_steps_ts = online_train(agent_ts, student_buf, "hopper",
                                        step_offset=teacher_steps, warmup=0)
print(f"online phase: {online_steps_ts} env steps in {time.perf_counter()-t0:.0f}s")

EXPERIMENTS.setdefault("hopper", {})["ts"] = dict(
    hist=hist_ts, teacher_steps=teacher_steps, online_steps=online_steps_ts,
    offline_updates=offline_updates, best_offline_f=best_off_f)

sts = steps_to_target(hist_ts)
print(f"\nteacher-student: steps-to-{TARGET_F} = "
      f"{sts if sts is not None else f'not reached (best {max(hist_ts["eval_f"]):.3f})'}")

In [ ]:
# === Run B (hopper): baseline — BPG only, from scratch ==========================
np.random.seed(SEED); torch.manual_seed(SEED)
agent_bl = BPGBC(11, 3, n_obj=4)              # bc_coef stays 0 -> plain BPG
buf_bl = ReplayBuffer(200_000, 11, 3, 4)

t0 = time.perf_counter()
hist_bl, online_steps_bl = online_train(agent_bl, buf_bl, "hopper",
                                        step_offset=0, warmup=1000)
print(f"baseline: {online_steps_bl} env steps in {time.perf_counter()-t0:.0f}s")

EXPERIMENTS["hopper"]["bl"] = dict(hist=hist_bl, teacher_steps=0,
                                   online_steps=online_steps_bl,
                                   offline_updates=0, best_offline_f=None)

stb = steps_to_target(hist_bl)
print(f"\nBPG-only: steps-to-{TARGET_F} = "
      f"{stb if stb is not None else f'not reached (best {max(hist_bl["eval_f"]):.3f})'}")

In [ ]:
# === Hopper results: steps-to-0.8 (total AND online-BPG-only) + per-term curves =
rows = [("teacher-student (MPPI-FPL -> BPG)", hist_ts, teacher_steps, offline_updates),
        ("BPG only (baseline)",               hist_bl, 0,             0)]

print(f"{'method':36s}{'steps to 0.8':>14s}{'teacher steps':>15s}"
      f"{'BPG steps':>11s}{'offline upd':>13s}{'best eval f':>13s}")
for name, hist, tsteps, oupd in rows:
    st = steps_to_target(hist)
    st_str = f"{st:,}" if st is not None else "not reached"
    bpg_at = (st - tsteps) if st is not None else None
    bpg_str = f"{bpg_at:,}" if bpg_at is not None else "-"
    print(f"{name:36s}{st_str:>14s}{tsteps:>15,}{bpg_str:>11s}"
          f"{oupd:>13,}{max(hist['eval_f']):>13.3f}")
print(f"\nteacher's own fulfillment (reference): {TEACHER_FS['hopper']:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

# --- left: eval fulfillment vs TOTAL env steps -----------------------------------
ax = axes[0]
ax.plot(hist_ts["eval_step"], hist_ts["eval_f"], "-o", ms=4, color="tab:blue",
        label="teacher-student")
ax.plot(hist_bl["eval_step"], hist_bl["eval_f"], "-o", ms=4, color="tab:red",
        label="BPG only")
ax.axhline(TARGET_F, color="k", ls="--", lw=1, alpha=0.6)
ax.axhline(TEACHER_FS["hopper"], color="gray", ls=":", lw=1.2, alpha=0.8)
ax.text(0.99, TEACHER_FS["hopper"], f" teacher {TEACHER_FS['hopper']:.2f}",
        va="bottom", ha="right", transform=ax.get_yaxis_transform(),
        fontsize=8, color="gray")
ax.axvline(teacher_steps, color="tab:blue", ls=":", lw=1.2, alpha=0.7)
ax.text(teacher_steps, 0.02, " teacher -> online", rotation=90, va="bottom",
        fontsize=8, color="tab:blue", alpha=0.9)
ax.set_xlabel("total env steps (teacher demos + online BPG)")
ax.set_ylabel(f"eval fulfillment  (mean u, p={FPL_P})")
ax.set_ylim(0, 1)
ax.set_title("hopper: fulfillment vs total experience")
ax.legend(loc="lower right")
ax.grid(alpha=0.3)

# --- right: per-term fulfillment for the teacher-student run ---------------------
ax = axes[1]
terms_ts = np.asarray(hist_ts["eval_terms"])
for j, lbl in enumerate(FPLEnv("hopper").task.cost_term_names_f):
    ax.plot(hist_ts["eval_step"], terms_ts[:, j], lw=1.8,
            label=lbl.replace("_fulfillment", ""))
ax.axvline(teacher_steps, color="k", ls=":", lw=1, alpha=0.5)
ax.set_xlabel("total env steps")
ax.set_ylabel("per-term eval fulfillment")
ax.set_ylim(0, 1)
ax.set_title("hopper teacher-student: per-objective fulfillment")
ax.legend(loc="lower right", fontsize=9)
ax.grid(alpha=0.3)

fig.suptitle(f"MPPI-FPL teacher -> BPG student vs BPG-only — hopper "
             f"(p={FPL_P}, seed={SEED}, budget={TOTAL_BUDGET:,})", y=1.02)
fig.tight_layout()
plt.show()

# Part 3 — Early-stopped distillation + the paradigm on walker & g1_standup

**Early stopping with a student checkpoint.** The first hopper run exposed the
classic offline actor-critic failure mode: distillation *peaked* at 0.849 eval
fulfillment after 2k updates, then decayed to 0.700 by the time the plateau rule
fired at 8k — the student left the offline phase well below its own best.
`offline_train` now (a) snapshots the full student (actor, critic, and both
target networks) at every eval that sets a new best, (b) stops early on the same
plateau rule, and (c) **restores the best checkpoint** before the online phase
begins.

**Two more environments.** The same teacher→student pipeline runs on:

- **`walker`** — planar biped, same 4 fulfillment terms as hopper but with
  `target_velocity = 1.5 m/s`: high fulfillment requires actually *walking*.
  Teacher = K=256, 2 iterations, noise 0.3, cubic/8 (probes at 0.81 over full
  400-step episodes; the tuned fast config survives but only tracks ≈ 0.74–0.76).
  Termination at torso z < 0.8.
- **`g1_standup`** — Unitree G1 humanoid (obs 69-d, actions 29-d), starting from
  the `stand` keyframe (the `run.py` convention), so the task is to *keep*
  standing: **3** fulfillment terms (orientation, height, joint-nominal) — the
  student's critic head shrinks to `n_obj = 3` accordingly. Teacher = K=256,
  2 iterations, noise 0.2, cubic/8, probing at 0.87. Termination at torso z < 0.5.

**A metric fix the probes forced.** The G1 stands *passively* (zero torque holds
fulfillment 0.95 indefinitely), and a falling policy still averages ≈ 0.92 over
its surviving steps — so per-surviving-step fulfillment is degenerate.
`episode_fulfillment` therefore normalizes by the **full horizon**: a terminated
episode scores 0 for every missing step. The same honesty applies to the
teachers: hopper's needed a 0.5 termination floor (recoverable crouch wobbles dip
below 0.7), and walker/g1 needed the stronger MPPI configs above to clear 0.8.

**Step accounting in the results.** Every curve and table reports *both* axes
requested: **total env steps** (teacher demo steps + online BPG steps — offline
updates are free) and **online BPG steps alone** (the student's own environment
interaction). The baseline has no teacher, so its two step counts coincide.

**A single-seed caveat.** Each number below is one seed (0). The paper reports
this metric as a *violin plot over 10 seeds* — the full distribution of
steps-to-threshold — because single runs are noisy (our first hopper baseline
crossed 0.8 in one jump from 0.632). Treat gaps smaller than a few thousand
steps as within noise.

In [ ]:
# === Generic experiment runners (same pipeline the hopper cells ran inline) =====
def run_teacher_student(env_name, log=print):
    """Gate -> demonstrate -> distill (early stop + best checkpoint) -> online BPG."""
    TEACHER_FS[env_name] = teacher_gate(env_name)
    buf, tsteps, _scores = collect_demos(env_name)
    env0 = FPLEnv(env_name)

    np.random.seed(SEED); torch.manual_seed(SEED)
    agent = BPGBC(env0.obs_dim, env0.act_dim, n_obj=env0.n_obj)
    t0 = time.perf_counter()
    evals, best_off_f, _ = offline_train(agent, buf, env_name, bc_coef=1.0, log=log)
    oupd = evals[-1][0] if evals else 0
    log(f"[{env_name}] offline: {oupd} updates in {time.perf_counter()-t0:.0f}s")

    t0 = time.perf_counter()
    hist, osteps = online_train(agent, buf, env_name,
                                step_offset=tsteps, warmup=0, log=log)
    log(f"[{env_name}] online: {osteps} env steps in {time.perf_counter()-t0:.0f}s")
    EXPERIMENTS.setdefault(env_name, {})["ts"] = dict(
        hist=hist, teacher_steps=tsteps, online_steps=osteps,
        offline_updates=oupd, best_offline_f=best_off_f)
    return agent


def run_bpg_only(env_name, log=print):
    """Baseline: BPG from scratch, same seed/budget/eval cadence."""
    env0 = FPLEnv(env_name)
    np.random.seed(SEED); torch.manual_seed(SEED)
    agent = BPGBC(env0.obs_dim, env0.act_dim, n_obj=env0.n_obj)
    buf = ReplayBuffer(200_000, env0.obs_dim, env0.act_dim, env0.n_obj)
    t0 = time.perf_counter()
    hist, osteps = online_train(agent, buf, env_name,
                                step_offset=0, warmup=1000, log=log)
    log(f"[{env_name}] baseline: {osteps} env steps in {time.perf_counter()-t0:.0f}s")
    EXPERIMENTS.setdefault(env_name, {})["bl"] = dict(
        hist=hist, teacher_steps=0, online_steps=osteps,
        offline_updates=0, best_offline_f=None)
    return agent


# ---- walker -----------------------------------------------------------------
_ = run_teacher_student("walker")
print()
_ = run_bpg_only("walker")

In [ ]:
# ---- g1_standup (69-d obs, 29-d actions, 3 fulfillment terms) -----------------
_ = run_teacher_student("g1_standup")
print()
_ = run_bpg_only("g1_standup")

In [ ]:
# === Combined results: all three envs, total steps AND online-BPG-only steps ====
ENVS_RUN = [n for n in ("hopper", "walker", "g1_standup") if n in EXPERIMENTS]

print(f"{'env':12s}{'method':17s}{'steps to 0.8 (total)':>22s}"
      f"{'teacher steps':>15s}{'BPG steps to 0.8':>18s}{'best eval f':>13s}")
for env_name in ENVS_RUN:
    for key, label in (("ts", "teacher-student"), ("bl", "BPG only")):
        r = EXPERIMENTS[env_name][key]
        st = steps_to_target(r["hist"])
        st_str = f"{st:,}" if st is not None else "not reached"
        bpg_str = f"{st - r['teacher_steps']:,}" if st is not None else "-"
        print(f"{env_name:12s}{label:17s}{st_str:>22s}{r['teacher_steps']:>15,}"
              f"{bpg_str:>18s}{max(r['hist']['eval_f']):>13.3f}")
print("\nteacher fulfillment reference:",
      {k: round(v, 3) for k, v in TEACHER_FS.items()})

fig, axes = plt.subplots(2, len(ENVS_RUN), figsize=(5 * len(ENVS_RUN), 7.6),
                         squeeze=False)
for col, env_name in enumerate(ENVS_RUN):
    ts, bl = EXPERIMENTS[env_name]["ts"], EXPERIMENTS[env_name]["bl"]

    # --- row 0: x = TOTAL env steps (teacher demos + online BPG) ---
    ax = axes[0, col]
    ax.plot(ts["hist"]["eval_step"], ts["hist"]["eval_f"], "-o", ms=3.5,
            color="tab:blue", label="teacher-student")
    ax.plot(bl["hist"]["eval_step"], bl["hist"]["eval_f"], "-o", ms=3.5,
            color="tab:red", label="BPG only")
    ax.axhline(TARGET_F, color="k", ls="--", lw=1, alpha=0.6)
    if env_name in TEACHER_FS:
        ax.axhline(TEACHER_FS[env_name], color="gray", ls=":", lw=1.2, alpha=0.8)
    ax.axvline(ts["teacher_steps"], color="tab:blue", ls=":", lw=1.2, alpha=0.7)
    ax.set_title(env_name)
    ax.set_xlabel("total env steps (teacher + BPG)")
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
    if col == 0:
        ax.set_ylabel(f"eval fulfillment (p={FPL_P})")
        ax.legend(loc="lower right", fontsize=9)

    # --- row 1: x = online BPG steps only (teacher demos excluded) ---
    ax = axes[1, col]
    ts_bpg_steps = np.asarray(ts["hist"]["eval_step"]) - ts["teacher_steps"]
    ax.plot(ts_bpg_steps, ts["hist"]["eval_f"], "-o", ms=3.5,
            color="tab:blue", label="teacher-student")
    ax.plot(bl["hist"]["eval_step"], bl["hist"]["eval_f"], "-o", ms=3.5,
            color="tab:red", label="BPG only")
    ax.axhline(TARGET_F, color="k", ls="--", lw=1, alpha=0.6)
    ax.set_xlabel("online BPG steps (student's own interaction)")
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
    if col == 0:
        ax.set_ylabel(f"eval fulfillment (p={FPL_P})")

fig.suptitle("MPPI-FPL → BPG distillation vs BPG-only — top: total experience, "
             f"bottom: student-only experience  (seed={SEED}, budget={TOTAL_BUDGET:,})",
             y=1.0)
fig.tight_layout()
plt.show()